# 🎯 [MOV02_M08] 도로 50곳 동시 차단 모의 대규모 보급 경로 재탐색 스트레스 테스트
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M08 |
| **미션 주제** | 도로 50곳 동시 차단 모의 대규모 보급 경로 재탐색 스트레스 테스트 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 도로 50곳 동시 차단 모의 대규모 보급 경로 재탐색 스트레스 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

도로 다수 차단 모의 보급 경로 재탐색 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **도로 50곳이 동시에 차단되는 상황을 1년치 일괄 배치로 모의해 보급 경로 재탐색의 성능과 안정성을 시험한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **192개 노드로 확장한 가상 도로망에서 매일 50개 구간을 무작위로 차단하고 365일 x 요청 8건을 자동 재탐색 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/pipeline/mov02_batch_runner.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 수작업 대비 업무 소요 시간 80% 이상 절감 효과

#### 미션 PASS 조건

- 정량 기준: 전체 배치 파이프라인 실행 시간 5분 이내, 자동화 성공률 99% 이상


- 진행 기준: 스케줄러 연동 테스트 및 자동 실행 확인


- 검증 방법: 실제 월간 업무 데이터 1년치 일괄 배치 시뮬레이션


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""확장한 가상 도로망에서 매일 도로 50곳이 동시에 차단되는 상황을 1년치(365일 x 요청 8건) 일괄 배치로 모의해 보급 경로를 자동 재탐색하고, 실행 시간·자동화 성공률·우회 거리 증가·수작업 대비 절감 효과를 측정하는 스트레스 테스트를 수행한다."""
import heapq
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network(seed=7, rows=12, cols=16)          # 스트레스용 확장 도로망(192 노드)
threats = make_threats(seed=9, n=6, xlim=(10, 70), ylim=(5, 50))
for e in edges:
    e["risk"] = edge_risk(e, nodes, threats)
EDGE_IDX = {e["id"]: e for e in edges}
DEPOT = "N0000"
DAYS, REQ_PER_DAY, N_BLOCK = 365, 8, 50
MANUAL_MIN = 12.0      # 가정: 통제관이 지도·보고서로 경로 1건을 수작업 재탐색하는 시간(분)
REVIEW_MIN = 1.0       # 가정: 자동 결과를 사람이 확인하는 시간(분)


def weight(e, gross=15):
    return edge_minutes(e) + 150 * e["risk"] if passable(e, gross) else None


def run_day(day, rng):
    """하루치 배치: 차단 보고 반영 -> 요청별 재탐색 -> 결과 기록. 오류는 건별로 격리해 기록."""
    reports = list(rng.choice(list(EDGE_IDX), N_BLOCK, replace=False))
    if day % 30 == 7:
        reports[0] = "E9" + reports[0][1:]                       # 오류 주입: 존재하지 않는 구간 번호(월 1회)
    if day % 45 == 11:
        reports.append(None)                                      # 오류 주입: 빈 보고(손상 레코드)
    bad = 0
    for e in edges:
        e["open"] = True
    for rid in reports:
        if rid in EDGE_IDX:
            EDGE_IDX[rid]["open"] = False
        else:
            bad += 1                                              # 잘못된 보고는 반영 보류 + 확인 요청 로그
    base_adj = build_adj(nodes, edges, lambda e: edge_minutes(e) + 150 * e["risk"])
    adj = build_adj(nodes, edges, weight)
    rows = []
    for k in range(REQ_PER_DAY):
        dest = f"N{int(rng.integers(3, 12)):02d}{int(rng.integers(8, 16)):02d}"
        try:
            c0, _, _ = dijkstra(base_adj, DEPOT, dest)
            c1, path, es = dijkstra(adj, DEPOT, dest)
            status = "재탐색 성공" if path else "도달 불가(공중수송 검토 요청)"
            rows.append({"day": day, "dest": dest, "status": status, "ok": True, "base": c0, "new": c1 if path else np.nan,
                         "km": sum(e["km"] for e in es) if path else np.nan, "bad_report": bad})
        except Exception as ex:                                   # 크래시 대신 실패로 기록
            rows.append({"day": day, "dest": dest, "status": f"실패 {ex!r}", "ok": False, "base": np.nan, "new": np.nan, "km": np.nan, "bad_report": bad})
    return rows


rng = np.random.default_rng(2024)
print(f"[교육용가상자료] 도로망 노드 {len(nodes)}개, 구간 {len(edges)}개 / 매일 {N_BLOCK}곳 동시 차단({N_BLOCK / len(edges):.0%}) x {DAYS}일 x 요청 {REQ_PER_DAY}건")
t0 = time.perf_counter()
out, marks = [], {int(DAYS * q) for q in (0.25, 0.5, 0.75)}
for day in range(DAYS):
    out += run_day(day, rng)
    if day + 1 in marks:
        print(f"  진행률 {(day + 1) / DAYS:.0%} ({day + 1}일 처리, 경과 {time.perf_counter() - t0:.1f}초)")
elapsed = time.perf_counter() - t0
r = pd.DataFrame(out)
print(f"  진행률 100% -> 완료 알림: 배치 {len(r)}건 처리, 실행 시간 {elapsed:.1f}초 (목표 5분 이내: {'달성' if elapsed < 300 else '미달'})")

succ = r["ok"].mean()
print("\n[처리 결과]")
print(r["status"].value_counts().to_string())
print(f"자동화 성공률(오류 없이 판정 완료) {succ:.2%} (목표 99% 이상: {'달성' if succ >= 0.99 else '미달'})")
print(f"손상·오기 차단 보고 {int(r.groupby('day')['bad_report'].first().sum())}건은 반영 보류 후 확인 요청으로 처리, 배치 중단 0건")
found = r[r["status"] == "재탐색 성공"]
inc = (found["new"] / found["base"] - 1)
print(f"우회로 인한 소요(가중) 증가: 평균 {inc.mean():.1%}, 상위 5% {inc.quantile(0.95):.1%}, 도달 불가 비율 {(r['status'] != '재탐색 성공').mean():.1%}")

manual = len(r) * MANUAL_MIN
auto = elapsed / 60 + len(r) * REVIEW_MIN
print(f"\n[수작업 대비] 수작업 {manual / 60:.0f}시간 vs 자동+검토 {auto / 60:.0f}시간 -> {1 - auto / manual:.1%} 절감 (목표 80% 이상: {'달성' if 1 - auto / manual >= 0.8 else '미달'})")
print("  (수작업 12분/건, 검토 1분/건은 교육용 가정값이며 실제 부대 측정치가 아님)")

# 차단 규모를 늘려 한계점 확인
print("\n[차단 규모별 한계점(각 30일 모의)]")
lim = []
for nb in (10, 50, 100, 150):
    N_BLOCK = nb
    rr = pd.DataFrame(sum((run_day(d, np.random.default_rng(d)) for d in range(30)), []))
    lim.append({"차단 수": nb, "차단 비율": f"{nb / len(edges):.0%}", "재탐색 성공": f"{(rr['status'] == '재탐색 성공').mean():.1%}", "자동화 성공률": f"{rr['ok'].mean():.1%}"})
print(pd.DataFrame(lim).to_string(index=False))
print("차단이 늘어도 판정 자체는 실패 없이 끝나며, 도달 불가 건은 통제관 검토 대상으로 넘긴다.")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
daily = r.groupby("day").apply(lambda g: (g["status"] == "재탐색 성공").mean())
ax[0].plot(daily.values, lw=0.8); ax[0].set_ylim(0, 1.05); ax[0].set_title("Daily reroute success (50 roads closed)"); ax[0].set_xlabel("day")
ax[1].hist(inc.dropna() * 100, bins=40, color="tab:orange"); ax[1].set_title("Detour cost increase (%)")
ax[2].bar(["manual", "auto+review"], [manual / 60, auto / 60], color=["gray", "tab:green"]); ax[2].set_ylabel("hours / year"); ax[2].set_title("Workload comparison")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 도로망 노드 192개, 구간 397개 / 매일 50곳 동시 차단(13%) x 365일 x 요청 8건
  진행률 25% (91일 처리, 경과 0.4초)
  진행률 50% (182일 처리, 경과 0.8초)
  진행률 75% (273일 처리, 경과 1.2초)
  진행률 100% -> 완료 알림: 배치 2920건 처리, 실행 시간 1.7초 (목표 5분 이내: 달성)

[처리 결과]
status
재탐색 성공               2845
도달 불가(공중수송 검토 요청)      75
자동화 성공률(오류 없이 판정 완료) 100.00% (목표 99% 이상: 달성)
손상·오기 차단 보고 20건은 반영 보류 후 확인 요청으로 처리, 배치 중단 0건
우회로 인한 소요(가중) 증가: 평균 5.3%, 상위 5% 16.6%, 도달 불가 비율 2.6%

[수작업 대비] 수작업 584시간 vs 자동+검토 49시간 -> 91.7% 절감 (목표 80% 이상: 달성)
  (수작업 12분/건, 검토 1분/건은 교육용 가정값이며 실제 부대 측정치가 아님)

[차단 규모별 한계점(각 30일 모의)]
 차단 수 차단 비율 재탐색 성공 자동화 성공률
   10    3% 100.0%  100.0%
   50   13% 100.0%  100.0%
  100   25%  84.6%  100.0%
  150   38%  61.7%  100.0%
차단이 늘어도 판정 자체는 실패 없이 끝나며, 도달 불가 건은 통제관 검토 대상으로 넘긴다.
```
</details>
